# MASCOTS Sweep Analysis

This notebook summarizes the results of the counterfactual search sweep across Chronos models (Tiny, Small, Base) for multiple datasets.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style
sns.set_theme(style="whitegrid")

## 1. Load Data
Loading results from both Electricity and Exchange Rate experiments.

In [ ]:
DATASETS = {
    "Electricity": "experiments_results/electricity/sweep_summary.csv",
    "Exchange Rate": "experiments_results/exchange_rate/sweep_summary.csv"
}

dfs = []
for name, path in DATASETS.items():
    if os.path.exists(path):
        d = pd.read_csv(path)
        d["Dataset"] = name
        dfs.append(d)
        print(f"Loaded {name}: {len(d)} rows")
    else:
        print(f"Warning: {path} not found")

if dfs:
    df = pd.concat(dfs, ignore_index=True)
    display(df.head())
else:
    print("No data loaded.")

## 2. Success Rates
Comparing the success rate of finding counterfactuals across Models and Datasets.

In [ ]:
# Calculate Success Rate per Model and Dataset
success_stats = df.groupby(["Dataset", "Model"])["IsSuccess"].mean().reset_index()
success_stats["Success Rate (%)"] = success_stats["IsSuccess"] * 100

plt.figure(figsize=(10, 6))
sns.barplot(data=success_stats, x="Model", y="Success Rate (%)", hue="Dataset", palette="viridis")
plt.title("Success Rate by Model and Dataset")
plt.ylim(0, 100)
plt.legend(title="Dataset")
plt.show()

print(success_stats)

## 3. Perturbation Cost (MSE)
For successful counterfactuals, how much did we have to change the input history? Lower is better.

In [ ]:
# Filter for successes only
df_success = df[df["IsSuccess"] == True].copy()

if not df_success.empty:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df_success, x="Model", y="Cost", hue="Dataset", palette="Set2")
    plt.yscale("log")
    plt.title("Distribution of Perturbation Cost (MSE) - Log Scale")
    plt.ylabel("Mean Squared Error (Log)")
    plt.legend(title="Dataset")
    plt.show()
else:
    print("No successful counterfactuals to plot cost for.")

## 4. Computation Time

In [ ]:
plt.figure(figsize=(12, 6))
sns.violinplot(data=df, x="Model", y="Time", hue="Dataset", palette="coolwarm", split=False)
plt.title("Computation Time Distribution per Window")
plt.ylabel("Time (Seconds)")
plt.legend(title="Dataset")
plt.show()

# Summary Table
summary = df.groupby(["Dataset", "Model"]).agg(
    Success_Rate=('IsSuccess', 'mean'),
    Mean_Cost=('Cost', lambda x: x[df['IsSuccess']].mean()),
    Median_Cost=('Cost', lambda x: x[df['IsSuccess']].median()),
    Mean_Time=('Time', 'mean')
).round(4)

display(summary)